# Measure Sandbox Widget

This notebook provides an interactive widget to:
1. Select a measure
2. Select one or more dimensions
3. Return a tabular DAX result
4. Control the order of returned columns
5. Temporarily edit the measure expression and format string
6. Revert or write changes back to the model

Draft edits are kept in the widget until you choose **Write Back To Model**.

In [ ]:
# Install required packages for this Fabric notebook
%pip install -q semantic-link-labs semantic-link-sempy anywidget traitlets pandas

In [ ]:
# Configure target semantic model
DATASET = "card_udf"
WORKSPACE = "Demo"  # or "Your Workspace"

In [ ]:
import json
import re
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
from IPython.display import display

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "sempy_labs").exists()),
    None,
 )
if repo_root is not None:
    src_path = str(repo_root / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

import anywidget
import traitlets

from sempy_labs.tom import connect_semantic_model
from sempy_labs._dax import evaluate_dax_impersonation
from sempy_labs._helper_functions import format_dax_object_name

In [ ]:
MEASURE_REF_PATTERN = re.compile(r"^'(?P<table>.+)'\[(?P<measure>.+)\]$")


def parse_measure_ref(measure_ref: str) -> Tuple[str, str]:
    match = MEASURE_REF_PATTERN.match(measure_ref)
    if not match:
        raise ValueError(f"Invalid measure reference: {measure_ref}")
    return match.group("table"), match.group("measure")


def load_model_metadata(dataset: str, workspace: Optional[str] = None) -> Dict[str, Dict]:
    measure_expr_by_ref: Dict[str, str] = {}
    measure_format_string_by_ref: Dict[str, str] = {}
    dimension_refs: List[str] = []

    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        for measure in tom.all_measures():
            if getattr(measure, "IsHidden", False):
                continue
            table_name = measure.Parent.Name
            ref = format_dax_object_name(table=table_name, column=measure.Name)
            measure_expr_by_ref[ref] = measure.Expression or ""
            measure_format_string_by_ref[ref] = (getattr(measure, "FormatString", None) or "")

        for column in tom.all_columns():
            if getattr(column, "IsHidden", False):
                continue
            table_name = column.Parent.Name
            ref = format_dax_object_name(table=table_name, column=column.Name)
            dimension_refs.append(ref)

    return {
        "measure_expr_by_ref": dict(sorted(measure_expr_by_ref.items())),
        "measure_format_string_by_ref": dict(sorted(measure_format_string_by_ref.items())),
        "dimension_refs": sorted(set(dimension_refs)),
    }


def build_dax_query(measure_ref: str, dimension_refs: List[str], topn: int = 200) -> str:
    if not dimension_refs:
        return f"EVALUATE ROW(\"Value\", {measure_ref})"

    dim_block = ",\n        ".join(dimension_refs)
    return f"""
EVALUATE
TOPN(
    {topn},
    SUMMARIZECOLUMNS(
        {dim_block},
        \"Value\", {measure_ref}
    ),
    [Value], DESC
)
""".strip()


def preview_measure_table(dataset: str, workspace: Optional[str], measure_ref: str, dimension_refs: List[str], topn: int) -> pd.DataFrame:
    dax_query = build_dax_query(measure_ref=measure_ref, dimension_refs=dimension_refs, topn=topn)
    return evaluate_dax_impersonation(
        dataset=dataset,
        dax_query=dax_query,
        workspace=workspace,
    )


def update_measure_definition(
    dataset: str,
    workspace: Optional[str],
    measure_ref: str,
    expression: str,
    format_string: Optional[str] = None,
) -> None:
    table_name, measure_name = parse_measure_ref(measure_ref)
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=False) as tom:
        # Prefer the TOMWrapper API first; fall back to direct mutation for older wrappers.
        if hasattr(tom, "update_measure"):
            update_kwargs = {"measure_name": measure_name, "expression": expression}
            if format_string is not None:
                update_kwargs["format_string"] = format_string
            try:
                tom.update_measure(**update_kwargs)
                return
            except TypeError:
                # Compatibility path for wrappers that require table_name.
                compat_kwargs = {"table_name": table_name, "measure_name": measure_name, "expression": expression}
                if format_string is not None:
                    compat_kwargs["format_string"] = format_string
                tom.update_measure(**compat_kwargs)
                return

        measure = tom.model.Tables[table_name].Measures[measure_name]
        measure.Expression = expression
        if format_string is not None:
            measure.FormatString = format_string

In [ ]:
metadata = load_model_metadata(dataset=DATASET, workspace=WORKSPACE)
measure_expr_by_ref = metadata["measure_expr_by_ref"]
measure_format_string_by_ref = metadata["measure_format_string_by_ref"]
dimension_refs = metadata["dimension_refs"]

if not measure_expr_by_ref:
    raise ValueError("No visible measures found in the selected semantic model.")

_WIDGET_CSS = """
.muw-root {
    --bg-solid: #ffffff;
    --surface: rgba(255, 255, 255, 0.9);
    --surface-2: rgba(0, 0, 0, 0.03);
    --border: rgba(0, 0, 0, 0.10);
    --border-strong: rgba(0, 0, 0, 0.18);
    --text: #1d1d1f;
    --text-secondary: #6e6e73;
    --accent: #007AFF;
    --accent-hover: #0a6cdb;
    --accent-soft: rgba(0, 122, 255, 0.12);
    --warning: #FF9500;
    --success: #34c759;
    --danger: #ff3b30;
    --radius: 14px;
    --radius-sm: 8px;
    --shadow: 0 1px 2px rgba(0,0,0,0.04), 0 8px 24px rgba(0,0,0,0.06);

    font-family: -apple-system, BlinkMacSystemFont, 'SF Pro Text', 'Helvetica Neue', Arial, sans-serif;
    color: var(--text);
    background: var(--bg-solid);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    padding: 18px;
    max-width: 1080px;
    box-sizing: border-box;
}

@media (prefers-color-scheme: dark) {
    .muw-root.muw-auto {
        --bg-solid: #1c1c1e;
        --surface: rgba(255, 255, 255, 0.05);
        --surface-2: rgba(255, 255, 255, 0.04);
        --border: rgba(255, 255, 255, 0.10);
        --border-strong: rgba(255, 255, 255, 0.18);
        --text: #f5f5f7;
        --text-secondary: #a1a1a6;
        --accent-soft: rgba(10, 132, 255, 0.18);
        --accent: #0A84FF;
    }
}

.muw-root.muw-dark {
    --bg-solid: #1c1c1e;
    --surface: rgba(255, 255, 255, 0.05);
    --surface-2: rgba(255, 255, 255, 0.04);
    --border: rgba(255, 255, 255, 0.10);
    --border-strong: rgba(255, 255, 255, 0.18);
    --text: #f5f5f7;
    --text-secondary: #a1a1a6;
    --accent-soft: rgba(10, 132, 255, 0.18);
    --accent: #0A84FF;
}

.muw-root * { box-sizing: border-box; }
.muw-root.muw-busy { opacity: 0.6; pointer-events: none; }

.muw-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 12px;
    gap: 8px;
}

.muw-title {
    font-size: 18px;
    font-weight: 600;
}

.muw-subtitle {
    font-size: 12px;
    color: var(--text-secondary);
}

.muw-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 12px;
}

@media (max-width: 900px) {
    .muw-grid { grid-template-columns: 1fr; }
}

.muw-card {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: var(--radius-sm);
    padding: 10px;
}

.muw-field {
    display: flex;
    flex-direction: column;
    gap: 6px;
    margin-bottom: 10px;
}

.muw-label {
    font-size: 12px;
    color: var(--text-secondary);
}

.muw-input, .muw-select, .muw-textarea {
    width: 100%;
    border: 1px solid var(--border-strong);
    background: var(--bg-solid);
    color: var(--text);
    border-radius: 10px;
    padding: 8px;
    font-size: 13px;
}
.muw-select[multiple] {
    min-height: 180px;
}

.muw-textarea {
    min-height: 180px;
    font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
    resize: vertical;
}

.muw-actions {
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
}

.muw-btn {
    border: 1px solid var(--border-strong);
    border-radius: 999px;
    padding: 7px 14px;
    cursor: pointer;
    font-size: 12px;
    font-weight: 600;
    transition: all 120ms ease;
    color: var(--text);
    background: var(--surface-2);
}
.muw-btn:hover {
    border-color: var(--text-secondary);
}
.muw-btn:disabled {
    opacity: 0.45;
    cursor: not-allowed;
    border-color: var(--border-strong);
}
.muw-btn:disabled:hover {
    border-color: var(--border-strong);
}
.muw-btn-primary {
    background: var(--accent);
    color: #fff;
    border-color: var(--accent);
}
.muw-btn-primary:hover {
    background: var(--accent-hover);
    border-color: var(--accent-hover);
}
.muw-btn-warning {
    background: var(--warning);
    color: #fff;
    border-color: var(--warning);
}
.muw-btn-danger {
    background: var(--danger);
    color: #fff;
    border-color: var(--danger);
}
.muw-btn-success {
    background: var(--success);
    color: #fff;
    border-color: var(--success);
}
.muw-status {
    display: none;
    margin-top: 10px;
    padding: 8px 10px;
    border-radius: var(--radius-sm);
    font-size: 12px;
}
.muw-status.show { display: block; }
.muw-status.info { background: var(--accent-soft); color: var(--accent); }
.muw-status.success { background: rgba(52,199,89,0.14); color: var(--success); }
.muw-status.error { background: rgba(255,59,48,0.14); color: var(--danger); }
.muw-results-wrap {
    margin-top: 12px;
    overflow: auto;
    max-height: 420px;
    border: 1px solid var(--border);
    border-radius: var(--radius-sm);
}
.muw-results {
    width: 100%;
    border-collapse: collapse;
    font-size: 12px;
}
.muw-results th, .muw-results td {
    border-bottom: 1px solid var(--border);
    padding: 6px 8px;
    text-align: left;
    white-space: nowrap;
}
.muw-results th {
    position: sticky;
    top: 0;
    background: var(--surface);
    z-index: 1;
}
.muw-helper {
    font-size: 11px;
    color: var(--text-secondary);
}
"""

_WIDGET_JS = """
function esc(s) {
  return String(s ?? '')
    .replace(/&/g, '&amp;')
    .replace(/</g, '&lt;')
    .replace(/>/g, '&gt;')
    .replace(/\"/g, '&quot;');
}

function renderTable(cols, rows) {
  if (!cols || cols.length === 0) {
    return '<div class=\"muw-helper\">No preview yet.</div>';
  }

  const head = '<tr>' + cols.map(c => `<th>${esc(c)}</th>`).join('') + '</tr>';
  const body = (rows || []).map(r => {
    return '<tr>' + cols.map(c => `<td>${esc(r[c])}</td>`).join('') + '</tr>';
  }).join('');

  return `
    <div class=\"muw-results-wrap\">
      <table class=\"muw-results\">
        <thead>${head}</thead>
        <tbody>${body}</tbody>
      </table>
    </div>`;
}

function send(model, action) {
  model.set('pending_action', action);
  model.set('run', (model.get('run') || 0) + 1);
  model.save_changes();
}

function selectedValues(selectEl) {
  return Array.from(selectEl.selectedOptions).map(o => o.value);
}

export default {
  render({ model, el }) {
    const root = document.createElement('div');
    root.className = 'muw-root';
    el.appendChild(root);

    function applyTheme() {
      root.classList.remove('muw-auto', 'muw-dark');
      const dm = model.get('dark_mode');
      if (dm === true) root.classList.add('muw-dark');
      else root.classList.add('muw-auto');
    }

    function setBusy() {
      root.classList.toggle('muw-busy', !!model.get('busy'));
    }

    function setStatus() {
      const s = model.get('status') || { message: '', kind: 'info' };
      const status = root.querySelector('.muw-status');
      if (!status) return;
      status.className = `muw-status ${s.message ? 'show' : ''} ${s.kind || 'info'}`;
      status.textContent = s.message || '';
    }

    function renderUI() {
      const measureOptions = model.get('measure_options') || [];
      const dimensionOptions = model.get('dimension_options') || [];
      const selectedMeasure = model.get('selected_measure') || '';
      const selectedDims = new Set(model.get('selected_dimensions') || []);
      const hasDraft = !!model.get('has_draft');
      const expression = model.get('expression') || '';
      const formatString = model.get('format_string') || '';
      const columnOrderText = model.get('column_order_text') || '';
      const topn = model.get('topn') || 100;
      const cols = model.get('result_columns') || [];
      const rows = model.get('result_rows') || [];

      root.innerHTML = `
        <div class=\"muw-header\">
          <div>
            <div class=\"muw-title\">Measure Sandbox Editor</div>
            <div class=\"muw-subtitle\">Preview by dimensions, set output column order, test temporary DAX, then write back when ready.</div>
          </div>
        </div>

        <div class=\"muw-grid\">
          <div class=\"muw-card\">
            <div class=\"muw-field\">
              <label class=\"muw-label\">Measure</label>
              <select id=\"muw-measure\" class=\"muw-select\">
                ${measureOptions.map(m => `<option value=\"${esc(m)}\" ${m === selectedMeasure ? 'selected' : ''}>${esc(m)}</option>`).join('')}
              </select>
            </div>

            <div class=\"muw-field\">
              <label class=\"muw-label\">Dimensions (multi-select)</label>
              <select id=\"muw-dims\" class=\"muw-select\" multiple>
                ${dimensionOptions.map(d => `<option value=\"${esc(d)}\" ${selectedDims.has(d) ? 'selected' : ''}>${esc(d)}</option>`).join('')}
              </select>
              <div class=\"muw-helper\">Use Ctrl/Cmd + click to select multiple dimensions.</div>
            </div>

            <div class=\"muw-field\">
              <label class=\"muw-label\">Column order (comma separated)</label>
              <input id=\"muw-column-order\" class=\"muw-input\" value=\"${esc(columnOrderText)}\" placeholder=\"e.g. 'Date'[Year], 'Product'[Category], Value\" />
              <div class=\"muw-helper\">Unknown names are ignored. Remaining columns are appended automatically.</div>
            </div>

            <div class=\"muw-field\">
              <label class=\"muw-label\">Top N rows</label>
              <input id=\"muw-topn\" class=\"muw-input\" type=\"number\" min=\"10\" max=\"5000\" step=\"10\" value=\"${esc(topn)}\" />
            </div>

            <div class=\"muw-actions\">
              <button id=\"muw-preview\" class=\"muw-btn muw-btn-primary\">Preview Table</button>
            </div>
          </div>

          <div class=\"muw-card\">
            <div class=\"muw-field\">
              <label class=\"muw-label\">Temporary DAX expression</label>
              <textarea id=\"muw-expression\" class=\"muw-textarea\">${esc(expression)}</textarea>
            </div>

            <div class=\"muw-field\">
              <label class=\"muw-label\">Format string</label>
              <input id=\"muw-format\" class=\"muw-input\" value=\"${esc(formatString)}\" placeholder=\"e.g. $#,0.00;($#,0.00)\" />
            </div>

            <div class=\"muw-actions\">
              <button id=\"muw-apply\" class=\"muw-btn muw-btn-warning\">Use For Preview</button>
              <button id=\"muw-revert\" class=\"muw-btn muw-btn-danger\" ${hasDraft ? '' : 'disabled'}>Revert</button>
              <button id=\"muw-writeback\" class=\"muw-btn muw-btn-success\" ${hasDraft ? '' : 'disabled'}>Write Back To Model</button>
            </div>
          </div>
        </div>

        <div class=\"muw-status\"></div>
        ${renderTable(cols, rows)}
      `;

      const measureEl = root.querySelector('#muw-measure');
      const dimsEl = root.querySelector('#muw-dims');
      const topnEl = root.querySelector('#muw-topn');
      const columnOrderEl = root.querySelector('#muw-column-order');
      const exprEl = root.querySelector('#muw-expression');
      const formatEl = root.querySelector('#muw-format');
      const previewBtn = root.querySelector('#muw-preview');
      const applyBtn = root.querySelector('#muw-apply');
      const revertBtn = root.querySelector('#muw-revert');
      const writeBackBtn = root.querySelector('#muw-writeback');

      measureEl?.addEventListener('change', () => {
        send(model, { action: 'select_measure', measure_ref: measureEl.value });
      });

      previewBtn?.addEventListener('click', () => {
        send(model, {
          action: 'preview',
          measure_ref: measureEl?.value || '',
          selected_dimensions: selectedValues(dimsEl || { selectedOptions: [] }),
          topn: Number(topnEl?.value || 100),
          column_order_text: columnOrderEl?.value || '',
        });
      });

      applyBtn?.addEventListener('click', () => {
        send(model, {
          action: 'apply_temp',
          measure_ref: measureEl?.value || '',
          expression: exprEl?.value || '',
          format_string: formatEl?.value || '',
        });
      });

      revertBtn?.addEventListener('click', () => {
        send(model, {
          action: 'revert',
          measure_ref: measureEl?.value || '',
        });
      });

      writeBackBtn?.addEventListener('click', () => {
        send(model, {
          action: 'write_back',
          measure_ref: measureEl?.value || '',
          expression: exprEl?.value || '',
          format_string: formatEl?.value || '',
        });
      });

      setStatus();
      setBusy();
      applyTheme();
    }

    renderUI();

    model.on('change:status', () => setStatus());
    model.on('change:busy', () => setBusy());
    model.on('change:dark_mode', () => applyTheme());
    model.on('change:measure_options', renderUI);
    model.on('change:dimension_options', renderUI);
    model.on('change:selected_measure', renderUI);
    model.on('change:selected_dimensions', renderUI);
    model.on('change:column_order_text', renderUI);
    model.on('change:expression', renderUI);
    model.on('change:format_string', renderUI);
    model.on('change:topn', renderUI);
    model.on('change:result_columns', renderUI);
    model.on('change:result_rows', renderUI);
    model.on('change:has_draft', renderUI);
  }
};
"""


class MeasureSandboxWidget(anywidget.AnyWidget):
    _esm = _WIDGET_JS
    _css = _WIDGET_CSS

    measure_options = traitlets.List([]).tag(sync=True)
    dimension_options = traitlets.List([]).tag(sync=True)
    selected_measure = traitlets.Unicode("").tag(sync=True)
    selected_dimensions = traitlets.List([]).tag(sync=True)
    column_order_text = traitlets.Unicode("").tag(sync=True)
    expression = traitlets.Unicode("").tag(sync=True)
    format_string = traitlets.Unicode("").tag(sync=True)
    topn = traitlets.Int(100).tag(sync=True)
    result_columns = traitlets.List([]).tag(sync=True)
    result_rows = traitlets.List([]).tag(sync=True)
    status = traitlets.Dict({"message": "Ready", "kind": "info"}).tag(sync=True)
    pending_action = traitlets.Dict({}).tag(sync=True)
    run = traitlets.Int(0).tag(sync=True)
    busy = traitlets.Bool(False).tag(sync=True)
    dark_mode = traitlets.Bool(False).tag(sync=True)
    has_draft = traitlets.Bool(False).tag(sync=True)


initial_measure_ref = next(iter(measure_expr_by_ref.keys()))
widget = MeasureSandboxWidget(
    measure_options=list(measure_expr_by_ref.keys()),
    dimension_options=dimension_refs,
    selected_measure=initial_measure_ref,
    selected_dimensions=[],
    column_order_text="",
    expression=measure_expr_by_ref[initial_measure_ref],
    format_string=measure_format_string_by_ref.get(initial_measure_ref, ""),
    topn=100,
    result_columns=[],
    result_rows=[],
    dark_mode=False,
)

_saved_measure_expr_by_ref: Dict[str, str] = dict(measure_expr_by_ref)
_draft_measure_expr_by_ref: Dict[str, str] = {}
_saved_measure_format_string_by_ref: Dict[str, str] = dict(measure_format_string_by_ref)
_draft_measure_format_string_by_ref: Dict[str, str] = {}


def _effective_expression(measure_ref: str) -> str:
    return _draft_measure_expr_by_ref.get(measure_ref, _saved_measure_expr_by_ref[measure_ref])


def _effective_format_string(measure_ref: str) -> str:
    return _draft_measure_format_string_by_ref.get(
        measure_ref, _saved_measure_format_string_by_ref.get(measure_ref, "")
    )


def _set_status(message: str, kind: str = "info") -> None:
    widget.status = {"message": message, "kind": kind}


def _set_results(df: pd.DataFrame) -> None:
    widget.result_columns = list(df.columns)
    widget.result_rows = json.loads(df.fillna("").to_json(orient="records"))


def _sync_draft_state(measure_ref: Optional[str] = None) -> None:
    target_measure_ref = measure_ref or widget.selected_measure
    widget.has_draft = (
        target_measure_ref in _draft_measure_expr_by_ref
        or target_measure_ref in _draft_measure_format_string_by_ref
    )


def _refresh_model_metadata() -> None:
    refreshed_metadata = load_model_metadata(dataset=DATASET, workspace=WORKSPACE)
    refreshed_measure_expr_by_ref = refreshed_metadata["measure_expr_by_ref"]
    refreshed_measure_format_string_by_ref = refreshed_metadata["measure_format_string_by_ref"]
    refreshed_dimension_refs = refreshed_metadata["dimension_refs"]

    if not refreshed_measure_expr_by_ref:
        raise ValueError("No visible measures found after refreshing model metadata.")

    measure_expr_by_ref.clear()
    measure_expr_by_ref.update(refreshed_measure_expr_by_ref)
    measure_format_string_by_ref.clear()
    measure_format_string_by_ref.update(refreshed_measure_format_string_by_ref)

    _saved_measure_expr_by_ref.clear()
    _saved_measure_expr_by_ref.update(refreshed_measure_expr_by_ref)
    _saved_measure_format_string_by_ref.clear()
    _saved_measure_format_string_by_ref.update(refreshed_measure_format_string_by_ref)

    # Keep only drafts that still point to existing measures.
    for measure_ref in list(_draft_measure_expr_by_ref.keys()):
        if measure_ref not in refreshed_measure_expr_by_ref:
            _draft_measure_expr_by_ref.pop(measure_ref, None)
    for measure_ref in list(_draft_measure_format_string_by_ref.keys()):
        if measure_ref not in refreshed_measure_expr_by_ref:
            _draft_measure_format_string_by_ref.pop(measure_ref, None)

    dimension_refs[:] = refreshed_dimension_refs
    dimension_set = set(refreshed_dimension_refs)
    widget.measure_options = list(refreshed_measure_expr_by_ref.keys())
    widget.dimension_options = refreshed_dimension_refs
    widget.selected_dimensions = [ref for ref in widget.selected_dimensions if ref in dimension_set]
    _sync_draft_state(widget.selected_measure)


def _get_preview_measure_ref(measure_ref: str) -> str:
    alias_measure = "__MeasureSandboxPreviewValue"
    return format_dax_object_name(table=parse_measure_ref(measure_ref)[0], column=alias_measure)


def _parse_column_order_text(column_order_text: str) -> List[str]:
    if not column_order_text:
        return []
    return [column.strip() for column in column_order_text.split(",") if column.strip()]


def _apply_column_order(df: pd.DataFrame, column_order_text: str) -> Tuple[pd.DataFrame, str, List[str]]:
    requested = _parse_column_order_text(column_order_text)
    current_columns = list(df.columns)

    if not requested:
        return df, ", ".join(current_columns), []

    ordered_existing = [column for column in requested if column in current_columns]
    missing = [column for column in requested if column not in current_columns]
    remaining = [column for column in current_columns if column not in ordered_existing]

    final_order = ordered_existing + remaining if ordered_existing else current_columns
    return df.loc[:, final_order], ", ".join(final_order), missing


def _build_preview_query(measure_ref: str, selected_dimensions: List[str], topn: int) -> str:
    effective_expression = _effective_expression(measure_ref)
    preview_measure_ref = _get_preview_measure_ref(measure_ref)

    query_lines = [
        f"DEFINE MEASURE {preview_measure_ref} = {effective_expression}",
        build_dax_query(
            measure_ref=preview_measure_ref,
            dimension_refs=selected_dimensions,
            topn=max(10, int(topn)),
        ),
    ]
    return "\n".join(query_lines)


def _select_measure(measure_ref: str) -> None:
    if measure_ref not in _saved_measure_expr_by_ref:
        raise ValueError(f"Unknown measure: {measure_ref}")

    widget.selected_measure = measure_ref
    widget.expression = _effective_expression(measure_ref)
    widget.format_string = _effective_format_string(measure_ref)
    _sync_draft_state(measure_ref)
    _set_status("Measure selected.", "info")


def _preview(
    measure_ref: str,
    selected_dimensions: List[str],
    topn: int,
    column_order_text: str,
    default_column_order_from_dimensions: bool = True,
) -> None:
    dax_query = _build_preview_query(
        measure_ref=measure_ref,
        selected_dimensions=selected_dimensions,
        topn=topn,
    )
    result_df = evaluate_dax_impersonation(
        dataset=DATASET,
        dax_query=dax_query,
        workspace=WORKSPACE,
    )

    order_text = (column_order_text or "").strip()
    if not order_text and default_column_order_from_dimensions:
        default_order = [column for column in selected_dimensions if column in result_df.columns]
        default_order.extend(column for column in result_df.columns if column not in default_order)
        order_text = ", ".join(default_order)

    ordered_df, normalized_order_text, missing_columns = _apply_column_order(result_df, order_text)

    widget.selected_measure = measure_ref
    widget.selected_dimensions = selected_dimensions
    widget.column_order_text = normalized_order_text
    widget.topn = max(10, int(topn))
    widget.expression = _effective_expression(measure_ref)
    widget.format_string = _effective_format_string(measure_ref)
    _set_results(ordered_df)

    if missing_columns:
        _set_status(
            "Preview query executed successfully. Some requested columns were not returned: " + ", ".join(missing_columns),
            "info",
        )
        return

    _set_status("Preview query executed successfully.", "success")


def _apply_temp(measure_ref: str, expression: str, format_string: str) -> None:
    expr = (expression or "").strip()
    if not expr:
        raise ValueError("Expression cannot be empty.")

    fmt = (format_string or "").strip()
    _draft_measure_expr_by_ref[measure_ref] = expr
    _draft_measure_format_string_by_ref[measure_ref] = fmt
    widget.selected_measure = measure_ref
    widget.expression = expr
    widget.format_string = fmt
    _sync_draft_state(measure_ref)
    _set_status(
        "Draft expression and format string stored for preview only. Use Write Back To Model to persist changes.",
        "info",
    )


def _revert(measure_ref: str) -> None:
    if measure_ref not in _saved_measure_expr_by_ref:
        raise ValueError("Unknown measure selected.")

    if measure_ref not in _draft_measure_expr_by_ref and measure_ref not in _draft_measure_format_string_by_ref:
        _set_status("Use For Preview first before reverting.", "info")
        return

    _draft_measure_expr_by_ref.pop(measure_ref, None)
    _draft_measure_format_string_by_ref.pop(measure_ref, None)
    saved_expr = _saved_measure_expr_by_ref[measure_ref]
    saved_fmt = _saved_measure_format_string_by_ref.get(measure_ref, "")
    widget.selected_measure = measure_ref
    widget.expression = saved_expr
    widget.format_string = saved_fmt
    _sync_draft_state(measure_ref)
    _set_status("Draft discarded. Editor reset to the current model definition.", "success")


def _write_back(measure_ref: str, expression: str, format_string: str) -> None:
    if measure_ref not in _draft_measure_expr_by_ref and measure_ref not in _draft_measure_format_string_by_ref:
        _set_status("Use For Preview first before writing back to the model.", "info")
        return

    expr = (expression or "").strip()
    if not expr:
        raise ValueError("Expression cannot be empty.")

    fmt = (format_string or "").strip()
    update_measure_definition(
        dataset=DATASET,
        workspace=WORKSPACE,
        measure_ref=measure_ref,
        expression=expr,
        format_string=fmt,
    )

    _draft_measure_expr_by_ref.pop(measure_ref, None)
    _draft_measure_format_string_by_ref.pop(measure_ref, None)

    _refresh_model_metadata()
    target_measure_ref = measure_ref if measure_ref in _saved_measure_expr_by_ref else next(iter(_saved_measure_expr_by_ref.keys()))
    widget.selected_measure = target_measure_ref
    widget.expression = _effective_expression(target_measure_ref)
    widget.format_string = _effective_format_string(target_measure_ref)
    _sync_draft_state(target_measure_ref)
    _set_status("Measure definition and format string written back to the semantic model and metadata refreshed.", "success")


def _on_run(change):
    action_payload = dict(widget.pending_action or {})
    action = action_payload.get("action")
    if not action:
        return

    widget.busy = True
    try:
        if action == "select_measure":
            _select_measure(action_payload.get("measure_ref", ""))
        elif action == "preview":
            _preview(
                measure_ref=action_payload.get("measure_ref", widget.selected_measure),
                selected_dimensions=list(action_payload.get("selected_dimensions", [])),
                topn=int(action_payload.get("topn", widget.topn)),
                column_order_text=action_payload.get("column_order_text", widget.column_order_text),
            )
        elif action == "apply_temp":
            _apply_temp(
                measure_ref=action_payload.get("measure_ref", widget.selected_measure),
                expression=action_payload.get("expression", ""),
                format_string=action_payload.get("format_string", ""),
            )
        elif action == "revert":
            _revert(action_payload.get("measure_ref", widget.selected_measure))
        elif action == "write_back":
            _write_back(
                measure_ref=action_payload.get("measure_ref", widget.selected_measure),
                expression=action_payload.get("expression", widget.expression),
                format_string=action_payload.get("format_string", widget.format_string),
            )
        else:
            raise ValueError(f"Unknown action: {action}")
    except Exception as exc:
        _set_status(f"{exc}", "error")
    finally:
        widget.pending_action = {}
        widget.busy = False


widget.observe(_on_run, names=["run"])
display(widget)

